In [ ]:
import pandas as pd
import json
from pathlib import Path
from elasticsearch import Elasticsearch, helpers
from pprint import pprint

In [ ]:
# Setup Paths (Dynamic)
BASE_DIR = Path.cwd()                        
DATA_DIR = BASE_DIR / "IR2025" / "IR2025"    
CSV_PATH = DATA_DIR / "documents.csv"
JSONL_PATH = DATA_DIR / "documents.jsonl"

print(f"Data folder: {DATA_DIR}")
print(f"Input file:  {CSV_PATH.name}")
print(f"Output file: {JSONL_PATH.name}")

In [ ]:
# Load CSV File
try:
    df = pd.read_csv(CSV_PATH, encoding="utf-8")
except FileNotFoundError:
    raise SystemExit(f"File not found: {CSV_PATH}")
except Exception as e:
    raise SystemExit(f"Error reading CSV: {e}")

In [ ]:
# Display Columns and Document Count
print(f"CSV loaded successfully → {len(df)} documents found")
print(f"Columns detected: {list(df.columns)}")

In [ ]:
# Validate Required Columns

required_cols = {"ID", "Text"}
missing = required_cols - set(df.columns)
if missing:
    raise SystemExit(f"Missing expected columns: {missing}")

In [ ]:
# Convert to JSONL
records_written = 0
with open(JSONL_PATH, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        record = {
            "id": row["ID"],     
            "text": row["Text"]
        }
        json_line = json.dumps(record, ensure_ascii=False)
        f.write(json_line + "\n")
        records_written += 1

print(f"Converted {records_written} rows → JSONL format")
print(f"Output saved at: {JSONL_PATH.resolve()}")

In [ ]:
# Connect to Elasticsearch
es = Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic", ""),
    verify_certs=False  
)


In [ ]:
# Test Connection
if es.ping():
    print("Successfully connected to Elasticsearch")
else:
    print("Connection failed — check if Elasticsearch is running")

In [ ]:
# Define Index 
INDEX_NAME = "ir2025"  

In [ ]:
# Check index
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME, ignore_unavailable=True)
    print(f"Deleted existing index '{INDEX_NAME}'")

In [ ]:
# Define Index Settings with BM25
index_settings = {
    "settings": {
        "similarity": {
            "default": {"type": "BM25"}
        },
        "analysis": {
            "analyzer": {
                "default": {
                    "type": "standard"
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "id": {"type": "keyword"},
            "text": {"type": "text", "analyzer": "standard"}
        }
    }
}


In [ ]:
# Create Index
es.indices.create(index=INDEX_NAME, body=index_settings)
print(f"Created new index '{INDEX_NAME}' with BM25 similarity")

In [ ]:
# Verify Index Creation
pprint(es.indices.get(index=INDEX_NAME))

In [ ]:
# Bulk Indexing Function
def generate_actions(jsonl_path, index_name):
    
    with open(jsonl_path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            yield {
                "_index": index_name,
                "_id": i,  
                "_source": json.loads(line)
            }

In [ ]:
# Bulk Index Documents
actions = generate_actions(JSONL_PATH, INDEX_NAME)
success, _ = helpers.bulk(es, actions)
print(f"Successfully indexed {success} documents into '{INDEX_NAME}'")

In [ ]:
# Load Queries
QUERIES_PATH = DATA_DIR / "queries.csv"

df_queries = pd.read_csv(QUERIES_PATH, encoding="utf-8")
print("Loaded queries successfully")
print(df_queries.head())

In [ ]:
# Search Function
def search_query(query_text, k):

    resp = es.search(
        index=INDEX_NAME,
        query={"match": {"text": {"query": query_text}}},
        size=k
    )
    return resp["hits"]["hits"]

In [ ]:
# Generate Results Files for k = 20, 30, 50
for k in [20, 30, 50]:
    results_path = DATA_DIR / f"results_{k}.txt"
    with open(results_path, "w", encoding="utf-8") as f:
        for _, row in df_queries.iterrows():
            qid = str(row[0])
            qtext = str(row[1])
            results = search_query(qtext, k)
            for rank, hit in enumerate(results, start=1):
                docid = hit["_source"]["id"]
                score = hit["_score"]
                f.write(f"{qid} Q0 {docid} {rank} {score:.4f} BM25\n")
    print(f"Created results file: {results_path}")

In [ ]:
QRELS_PATH = DATA_DIR / "qrels.txt"

In [ ]:
# Load Qrels Function
def load_qrels(path):
    return pd.read_csv(path, sep=r"\s+", header=None, names=["qid", "_", "docid", "rel"])

In [ ]:
# Load Results Function
def load_results(path):
    return pd.read_csv(path, sep=r"\s+", header=None, names=["qid", "_", "docid", "rank", "score", "method"])


In [ ]:
# Precision@k Calculation
def precision_at_k(results, qrels, k):
    precisions = []
    for qid, group in results.groupby("qid"):
        rel_docs = set(qrels[qrels["qid"] == qid]["docid"])
        retrieved = group.sort_values("rank").head(k)
        hits = sum(doc in rel_docs for doc in retrieved["docid"])
        precisions.append(hits / k)
    return sum(precisions) / len(precisions)

In [ ]:
# Mean Average Precision Calculation
def mean_average_precision(results, qrels):
    APs = []
    for qid, group in results.groupby("qid"):
        rel_docs = set(qrels[qrels["qid"] == qid]["docid"])
        retrieved = group.sort_values("rank")
        hits, cum_prec = 0, 0.0
        for i, doc in enumerate(retrieved["docid"], 1):
            if doc in rel_docs:
                hits += 1
                cum_prec += hits / i
        if hits > 0:
            APs.append(cum_prec / hits)
    return sum(APs) / len(APs)

In [ ]:
qrels = load_qrels(QRELS_PATH)

In [ ]:
# Evaluate Results
for k in [20, 30, 50]:
    results = load_results(DATA_DIR / f"results_{k}.txt")
    print(f"\nEvaluation for results_{k}.txt")
    for kk in [5, 10, 15, 20]:
        p = precision_at_k(results, qrels, kk)
        print(f"Precision@{kk}: {p:.4f}")
    map_val = mean_average_precision(results, qrels)
    print(f"MAP: {map_val:.4f}")

In [ ]:
##### THA TO FTIKSW ME PINAKA KANONIKO KAI OXI AYTH THN AHDIA

# Visualization of Precision@k
import matplotlib.pyplot as plt
import seaborn as sns

# RESULTS from evaluation
data = {
    "Retrieval_k": [20, 30, 50],
    "P@5":  [0.82, 0.82, 0.82],
    "P@10": [0.71, 0.71, 0.71],
    "P@15": [0.5867, 0.5867, 0.5867],
    "P@20": [0.52, 0.52, 0.52],
    "MAP":  [0.8039, 0.7670, 0.7039]
}

df = pd.DataFrame(data)

sns.set(style="whitegrid", context="talk")
palette = sns.color_palette("crest", as_cmap=False)

fig, ax1 = plt.subplots(figsize=(10, 6))
precision_cols = ["P@5", "P@10", "P@15", "P@20"]
df_melted = df.melt(id_vars="Retrieval_k", value_vars=precision_cols,
                    var_name="Metric", value_name="Score")

sns.barplot(
    data=df_melted, x="Retrieval_k", y="Score", hue="Metric",
    palette=palette, ax=ax1
)

ax1.set_title("Precision@k for Different Retrieval Depths (BM25 Baseline)", fontsize=16, weight="bold")
ax1.set_xlabel("Number of Retrieved Documents (k)", fontsize=13)
ax1.set_ylabel("Precision", fontsize=13)
ax1.legend(title="Metric", loc="upper right")
ax1.set_ylim(0, 1)

for container in ax1.containers:
    ax1.bar_label(container, fmt="%.2f", label_type="edge", fontsize=10, padding=3)

plt.tight_layout()
plt.show()
